## Local Inference on GPU 
Model page: https://huggingface.co/Qwen/Qwen-VL

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen-VL)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
pip install qwen-vl-utils

In [ ]:
pip install qwen-vl-utils transformers accelerate torch torchvision

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

# 1. Load the model and processor
model_id = "Qwen/Qwen2-VL-7B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# 2. Prepare a simple test input
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# 3. Process and Generate
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

generated_ids = model.generate(**inputs, max_new_tokens=128)
output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
print(output_text)

In [ ]:
pip install bitsandbytes

In [3]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# 1. Setup 4-bit Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model_id = "Qwen/Qwen2-VL-7B-Instruct"

# 2. Load Model (It will now take ~5GB VRAM instead of 14GB)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# 3. Set Image Resolution Limits
# Qwen2-VL handles dynamic resolutions, but we need to cap it for T4s
min_pixels = 256 * 28 * 28 # Minimum resolution
max_pixels = 640 * 28 * 28  # Capping it here prevents the 12GB spike
processor = AutoProcessor.from_pretrained(
    model_id, 
    min_pixels=min_pixels, 
    max_pixels=max_pixels
)

# --- Rest of your generation code ---
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://iitandm.in/uploads/geo-tag/Landscaping%20with%20Trees%20and%20plants.png"},
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

generated_ids = model.generate(**inputs, max_new_tokens=128)
output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
print(output_text)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

['system\nYou are a helpful assistant.\nuser\nDescribe this image.\nassistant\nThe image depicts the campus of the Indian Engineering College (IES) in Bhopal, Madhya Pradesh, India. The photograph shows a well-maintained outdoor area with several palm trees lining a pathway. In the foreground, there is a large, bold "IES" sign made of white letters, which stands out against the greenery. The background includes a series of buildings that appear to be part of the college campus, with a mix of modern architectural elements and open spaces.\n\nTo the right of the image, there is a Google Maps marker indicating the location of the IES College Road, Bhopal, Madhya Pradesh,']
